# Notebook 17: CNNs, RNNs, and LSTMs
**DCS 404 — Data Science and Machine Learning**

---

## Learning Objectives
- Understand convolutional layers and why they suit images
- Build and train a CNN on MNIST
- Explain the vanishing gradient problem in plain RNNs
- Understand LSTM gating mechanisms
- Apply an LSTM to a sequence classification task

## Part A — Convolutional Neural Networks (CNN)

## 1. Why Not MLP for Images?

- A 28×28 image = 784 pixels; 100 hidden units → 78,400 parameters per layer
- **No spatial awareness**: shuffling pixels randomly leaves accuracy unchanged
- **Not translation invariant**: a cat in the corner vs the centre look completely different

## 2. Convolutional Layer

A **filter (kernel)** slides across the image computing dot products at each position:
- **Parameter sharing**: same filter applied everywhere → far fewer parameters
- **Translation equivariance**: a feature fires wherever the pattern appears
- **Stacking conv layers**: early layers detect edges, middle layers detect shapes, deep layers detect objects

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision, torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
# Manual 2D convolution to show edge detection
import numpy as np

img = np.zeros((28, 28))
img[5:23, 5:10] = 1.0
img[5:10, 5:23] = 1.0

vert_k  = np.array([[-1,0,1],[-1,0,1],[-1,0,1]])
horiz_k = np.array([[-1,-1,-1],[0,0,0],[1,1,1]])

def conv2d(img, k):
    kh, kw = k.shape
    out = np.zeros((img.shape[0]-kh+1, img.shape[1]-kw+1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i,j] = (img[i:i+kh, j:j+kw]*k).sum()
    return out

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, data, title in zip(axes,
    [img, conv2d(img, vert_k), conv2d(img, horiz_k)],
    ['Original', 'Vertical edge filter', 'Horizontal edge filter']):
    ax.imshow(data, cmap='gray'); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

## 3. CNN for MNIST

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_set = torchvision.datasets.MNIST('./data', train=True,  download=True, transform=transform)
test_set  = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)
train_dl  = DataLoader(train_set, batch_size=128, shuffle=True)
test_dl   = DataLoader(test_set,  batch_size=256, shuffle=False)
print(f'Train: {len(train_set)}, Test: {len(test_set)}, Image: {train_set[0][0].shape}')

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 28→14
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 14→7
        )
        self.clf = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 10)
        )
    def forward(self, x): return self.clf(self.features(x))

cnn = CNN().to(device)
print(cnn)
print(f'Parameters: {sum(p.numel() for p in cnn.parameters()):,}')

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn.parameters(), lr=1e-3)

train_accs, test_accs = [], []
for epoch in range(5):
    cnn.train()
    correct = total = 0
    for X, y in train_dl:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = cnn(X)
        loss = criterion(out, y)
        loss.backward(); optimizer.step()
        correct += (out.argmax(1)==y).sum().item(); total += len(y)
    train_accs.append(correct/total)

    cnn.eval()
    correct = total = 0
    with torch.no_grad():
        for X, y in test_dl:
            X, y = X.to(device), y.to(device)
            correct += (cnn(X).argmax(1)==y).sum().item(); total += len(y)
    test_accs.append(correct/total)
    print(f'Epoch {epoch+1}/5 | Train={train_accs[-1]:.4f} | Test={test_accs[-1]:.4f}')

## Part B — Recurrent Networks (RNN) and LSTM

## 4. Why RNNs?

Language, time series, and audio are **sequential** — order matters.

$$h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b)$$

The hidden state $h_t$ carries information forward. **Problem**: with long sequences, gradients either vanish (exponentially small) or explode (exponentially large) during backpropagation through time.

## 5. LSTM Gates

LSTMs introduce a **cell state** $C_t$ ("memory highway") with three gates:

| Gate | Controls |
|------|----------|
| **Forget** $f_t = \sigma(W_f[h_{t-1}, x_t])$ | What to erase from $C$ |
| **Input** $i_t = \sigma(W_i[h_{t-1}, x_t])$ | What new info to write |
| **Output** $o_t = \sigma(W_o[h_{t-1}, x_t])$ | What to expose as $h_t$ |

$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t, \quad h_t = o_t \odot \tanh(C_t)$$

In [ ]:
# LSTM for sequence classification
# Positive: upward trend; Negative: downward trend
np.random.seed(42); torch.manual_seed(42)

def make_seq_data(n=1000, L=20):
    X, y = [], []
    for _ in range(n):
        if np.random.rand() > 0.5:
            seq = np.cumsum(np.random.randn(L)) + np.random.randn(L)*0.3; y.append(1)
        else:
            seq = -np.cumsum(np.random.randn(L)) + np.random.randn(L)*0.3; y.append(0)
        X.append(seq)
    return (np.array(X, dtype=np.float32),
            np.array(y, dtype=np.float32))

X_s, y_s = make_seq_data()
X_tr = torch.tensor(X_s[:800]).unsqueeze(-1)  # (800, 20, 1)
X_te = torch.tensor(X_s[800:]).unsqueeze(-1)
y_tr = torch.tensor(y_s[:800]).unsqueeze(1)
y_te = torch.tensor(y_s[800:]).unsqueeze(1)

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_size=1, hidden=32, layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, layers, batch_first=True)
        self.fc   = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

lstm = LSTMClassifier().to(device)
opt = optim.Adam(lstm.parameters(), lr=1e-3)
crit = nn.BCEWithLogitsLoss()

from torch.utils.data import TensorDataset, DataLoader
dl = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)

for epoch in range(30):
    lstm.train()
    tot = 0
    for Xb, yb in dl:
        Xb, yb = Xb.to(device), yb.to(device)
        opt.zero_grad()
        l = crit(lstm(Xb), yb)
        l.backward(); opt.step()
        tot += l.item()
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1:3d} | Loss: {tot/len(dl):.4f}')

lstm.eval()
with torch.no_grad():
    preds = (torch.sigmoid(lstm(X_te.to(device))) > 0.5).float()
    acc = (preds == y_te.to(device)).float().mean()
print(f'Test Accuracy: {acc.item():.4f}')

## Exercises

1. Visualise the 32 feature maps from the first conv layer for a test image. What patterns do they detect?
2. Add a third conv block. Does it improve MNIST accuracy? At what cost?
3. Replace the LSTM with a GRU (`nn.GRU`). Compare parameters, speed, and accuracy.
4. Use a bidirectional LSTM (`bidirectional=True`). Does it help for the sequence classification task?

## Reflection Questions
- Max pooling reduces spatial size. What information is preserved? What is lost?
- An RNN uses the same weight matrix at every time step. Why is this both a strength and a weakness?
- Why do LSTM gates help with the vanishing gradient problem? (Hint: think about what the forget gate does to the gradient.)

---
**Next →** `18_stock_prediction_lstm.ipynb`